#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col, min, max, count, when, length 

#Reading from Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

# Silver Transformations

In [0]:
"""""""""
df.select(F.col("sls_ord_num")).distinct().count() 
df.count() #More sls_ord_num than rows, cant it be id?
df.dropDuplicates().count() #No duplicates
df.select([
    F.count(F.when(F.col(c).isNull(), True)).alias(c) 
    for c in df.columns
]).show() #7 sls price with NULL value, leave or delete?
df.groupBy(F.col("sls_ord_num")) \
  .count() \
  .orderBy(F.col("count").desc()) \
  .limit(10).show() #We have sls order number from 1 to 8


#Show orders where the sls_ord_num = SO58845
#df.filter(F.col("sls_quantity") == SO58845).display() #Jus notice sls_quantity and sls_price are basically the same, when one is null the other one is not
#df.filter(F.col("sls_price") != F.col("sls_sales")).display() #There are 1 cases where the values are not the same or are the negative evrsion of the other, select the positive value when there are negatives?

#The dates are int values and have the format yyyy-MM-dd but without the - -, format to date, a simple cast will work?
df.select(F.col("sls_due_dt"), F.col("sls_ship_dt"), F.col("sls_order_dt")) \
  .summary("count", "min", "25%", "50%", "75%", "max") \
  .show()

df.filter(F.col("sls_price") != F.col("sls_sales")).display()
"""""""""


Summary Data Exploration:
We have to do Trimming, should we create an id for each order?, merge sls_quantity and sls_price, when are the same no problem, is one is null take the other, if one is negative take the positive one, else put null values, cast date the format is missing the - - but is in the correct style, 

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Cleaning Dates

In [0]:
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8), None)
        .otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8), None)
        .otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8), None).
        otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)


##Sales and Price Corrections
In the column sales prices if sales price is null or negative (invalid), if sales quantity is different from 0 then divide sale sales/quantity and replace the null or negative value in the sales price column, remember that withColumn is telling us the value in the column given as first parameter will be replace with the second aprameter which in this case is the whole logic describe previuosly 

In [0]:
df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)

## Renaming columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity Checks Dataframe

In [0]:
df.limit(10).display()

# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

## Sanity Checks Silver Table 

In [0]:
%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 10